# Trả lời hàng loạt câu hỏi (không chấm điểm) · Qwen3-8B trên GPU Kaggle

Notebook này chỉ **chạy hệ thống để lấy câu trả lời** cho một file câu hỏi chưa có đáp án. Cấu hình giống các lượt hỏi–đáp đã đánh giá (graph luật + tội + văn bản hướng dẫn, không dùng án, Qwen3-8B Q8, temperature 0). Notebook đánh giá có đáp án là `qwen_kaggle.ipynb`.

## Chuẩn bị file câu hỏi
- CSV (UTF-8) hoặc Excel, mỗi câu một dòng, tối thiểu 2 cột: mã câu (không trùng) và nội dung câu hỏi. Mặc định tên cột là `id` và `cau_hoi` (đổi ở ô đầu tiên nếu khác).
- Tải lên Kaggle thành **Dataset riêng tư** (Create → New Dataset → Private), **không** đưa lên GitHub (repo công khai).

## Chạy
1. Import notebook này (File → Import Notebook).
2. **Add Input** → Datasets → Your Datasets → dataset câu hỏi.
3. **Settings**: Accelerator = GPU T4 x2, Internet = On.
4. **Save Version → Save & Run All (Commit)**: chạy nền tối đa 12 giờ, có thể tắt máy.
5. Kết quả ở tab **Output**: `outputs/answers_qwen3_8b-q8_0_<TAG>.xlsx` (mỗi câu một dòng, có cột trống để đánh giá) và `.json` (đầy đủ quá trình), log từng phần, `cache_llm.tar.gz`.

Bị cắt ở 12 giờ hoặc lỗi: **Add Input** thêm Output của version trước rồi chạy lại; cache được giải nén nên các câu đã xong trả lời ngay.

In [ ]:
# ===== CẤU HÌNH =====
INPUT_NAME = 'cau_hoi.csv'   # tên file câu hỏi trong dataset đã Add Input (.csv hoặc .xlsx)
ID_COL = 'id'                # cột mã câu
TEXT_COL = 'cau_hoi'         # cột nội dung câu hỏi
TITLE_COL = ''               # cột tiêu đề nếu có (đặt trước nội dung), để trống nếu không
TAG = 'fb'                   # hậu tố tên file kết quả: outputs/answers_qwen3_8b-q8_0_<TAG>.xlsx
LIMIT = 0                    # >0 = chỉ chạy N câu đầu để thử

In [ ]:
import glob, os, shutil, subprocess, sys, time, urllib.request
KAGGLE = os.path.exists('/kaggle')
!nvidia-smi --query-gpu=name,memory.total --format=csv

if KAGGLE:
    WORK = '/kaggle/working'
    OUT = WORK                       # outputs/ và cache_llm/ ở đây thành Output của version
    old_caches = glob.glob('/kaggle/input/**/cache_llm.tar.gz', recursive=True)
else:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content'
    OUT = '/content/drive/MyDrive/KhoaLuan'
    old_caches = []
# Kaggle: cache ngoài Output (hàng nghìn file nhỏ), gom thành cache_llm.tar.gz sau mỗi lượt
CACHE = '/kaggle/tmp/cache_llm' if KAGGLE else f'{OUT}/cache_llm'
os.makedirs(f'{OUT}/outputs', exist_ok=True)
os.makedirs(CACHE, exist_ok=True)
for old in old_caches:               # Kaggle: tiếp tục từ output của version trước
    subprocess.check_call(['tar', 'xzf', old, '-C', CACHE, '--skip-old-files'])
    print(f'Đã giải nén cache từ {old}')
print('Kaggle' if KAGGLE else 'Colab', '| cache:', len(os.listdir(CACHE)), 'lời gọi')

In [ ]:
BRANCH = 'claude/legalgraphrag-framework-hv23z3'
# Kaggle: ngoài /kaggle/working, để Output chỉ có kết quả (repo có hàng nghìn file cache)
REPO = '/kaggle/tmp/Agent-Graph-Rag' if KAGGLE else f'{WORK}/Agent-Graph-Rag'
!rm -rf {REPO} && git clone -q -b {BRANCH} https://github.com/bighousevn/Agent-Graph-Rag.git {REPO}
os.chdir(REPO)
!pip install -q -r requirements.txt
!ls -la outputs/hierargraph.pkl

In [ ]:
# File câu hỏi: tìm trong các Input đã gắn (Kaggle đặt dataset dưới /kaggle/input/...)
found = [p for p in glob.glob(f'/kaggle/input/**/{INPUT_NAME}', recursive=True)] if KAGGLE else []
assert found, f'Không thấy {INPUT_NAME} trong /kaggle/input: hãy Add Input dataset câu hỏi'
INPUT = found[0]
print('File câu hỏi:', INPUT)

## Biến môi trường

In [ ]:
MODEL = 'qwen3:8b-q8_0'  # Qwen3-8B như bài báo; Q8 (~9 GB) + ngữ cảnh 24k (~3,5 GB) vừa một GPU T4 16 GB
os.environ.update({
    'LLM_PROVIDER': 'ollama',
    'LLM_API_KEY': 'ollama',          # Ollama không cần key; biến này chỉ để chọn đúng nhà cung cấp
    'LLM_MODEL': MODEL,
    'LLM_TIMEOUT': '600',
    'LLM_TEMPERATURE': '0',          # tất định: chạy lại cùng câu ra cùng kết quả, so sánh giữa các lần mới rõ
    'CACHE_DIR': CACHE,
})
print('Đã đặt biến môi trường cho', MODEL)

## Ollama + Qwen3-8B

In [ ]:
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
# Một server Ollama cho mỗi GPU (Kaggle T4 x2): hai hàng lượt chạy song song.
import torch
N_GPU = max(1, torch.cuda.device_count())
PORTS = [11434 + i for i in range(N_GPU)]
servers = []
for i, port in enumerate(PORTS):
    env = {**os.environ, 'CUDA_VISIBLE_DEVICES': str(i), 'OLLAMA_HOST': f'127.0.0.1:{port}',
           'OLLAMA_CONTEXT_LENGTH': '24576', 'OLLAMA_KEEP_ALIVE': '-1'}
    servers.append(subprocess.Popen(['ollama', 'serve'], env=env, stdout=open(f'{OUT}/ollama{i}.log', 'w'), stderr=subprocess.STDOUT))
for port in PORTS:
    for _ in range(60):
        try:
            urllib.request.urlopen(f'http://127.0.0.1:{port}'); break
        except Exception:
            time.sleep(1)
!OLLAMA_HOST=127.0.0.1:11434 ollama pull {MODEL} 2>&1 | tail -1
print(N_GPU, 'GPU, cổng', PORTS)

In [ ]:
# Thử: trả lời ngay, không suy nghĩ, chạy trên GPU (dừng notebook nếu không đạt).
# Lần gọi đầu nạp model lên GPU (~1 phút); đo thời gian ở lần thứ hai.
sys.path.insert(0, REPO)
from vn_legal_graph.config import AppConfig
from vn_legal_graph.llm import LLMClient
from vn_legal_graph.judge.judge_law import parse_bool
for port in PORTS:
    os.environ['LLM_BASE_URL'] = f'http://127.0.0.1:{port}/v1'
    c = LLMClient(AppConfig.from_env_file('/nonexistent'))
    q = 'Trả lời đúng một từ true hoặc false: trộm cắp tài sản là tội phạm?'
    t = time.time()
    c._ollama_chat(q, 16)
    load = time.time() - t
    t = time.time()
    raw, truncated = c._ollama_chat(q, 16)
    took = time.time() - t
    print(port, repr(raw), f'nạp model {load:.0f}s, trả lời {took:.1f}s')
    assert not truncated and parse_bool(raw) is True, 'Qwen vẫn suy nghĩ hoặc không trả lời true/false'
    assert took < 15, f'Một câu true/false mất {took:.0f}s: có thể đang chạy trên CPU hoặc vẫn suy nghĩ'
!OLLAMA_HOST=127.0.0.1:11434 ollama ps

## Trả lời

Mỗi GPU trả lời một nửa số câu (`--shard 1/2`, `--shard 2/2`), ghi kết quả sau mỗi câu; kết quả và cache được lưu ra Output mỗi 15 phút. Khi cả hai xong, notebook gộp thành một file JSON + một file Excel.

In [ ]:
COMMAND = (f'python scripts/answer_batch.py --input "{INPUT}" --id-col {ID_COL} --text-col {TEXT_COL}'
           + (f' --title-col {TITLE_COL}' if TITLE_COL else '') + (f' --limit {LIMIT}' if LIMIT else '') + f' --tag {TAG}')
!{COMMAND} --dry-run
SHARDS = [f'{k}/{N_GPU}' for k in range(1, N_GPU + 1)] if N_GPU > 1 else ['']
import threading

def save():
    for f in glob.glob('outputs/answers_*'):
        shutil.copy(f, f'{OUT}/outputs/')
    if KAGGLE:
        subprocess.call(['tar', 'czf', f'{OUT}/cache_llm.tar.gz.tmp', '-C', CACHE, '.'])
        os.replace(f'{OUT}/cache_llm.tar.gz.tmp', f'{OUT}/cache_llm.tar.gz')

def log_of(shard):
    return f"{OUT}/outputs/tra_loi_{TAG}{'_phan' + shard.replace('/', '-') if shard else ''}.log"

def run_shard(gpu, shard):
    env = {**os.environ, 'LLM_BASE_URL': f'http://127.0.0.1:{PORTS[gpu]}/v1', 'PYTHONUNBUFFERED': '1'}
    t = time.time()
    cmd = COMMAND + (f' --shard {shard}' if shard else '')
    code_ = subprocess.call(f'{cmd} > {log_of(shard)} 2>&1', shell=True, env=env)
    print(f'GPU {gpu} | phần {shard or "toàn bộ"}: mã thoát {code_}, {(time.time() - t) / 60:.0f} phút', flush=True)

workers = [threading.Thread(target=run_shard, args=(g, s)) for g, s in enumerate(SHARDS)]
for w in workers:
    w.start()
while any(w.is_alive() for w in workers):  # lưu mỗi 15 phút: bị cắt ở 12 giờ vẫn còn kết quả
    for w in workers:
        w.join(timeout=900 / len(workers))
    save()
    done = {s or 'toàn bộ': sum(1 for l in open(log_of(s), encoding='utf-8') if l.startswith('['))
            for s in SHARDS if os.path.exists(log_of(s))}
    print(time.strftime('%H:%M'), 'đã xong (câu):', done, flush=True)

MODEL_SLUG = MODEL.replace(':', '_')
parts = [f'outputs/answers_{MODEL_SLUG}_{TAG}_phan{s.replace("/", "-")}.json' for s in SHARDS] if SHARDS != [''] else []
MERGED = f'outputs/answers_{MODEL_SLUG}_{TAG}.json'
subprocess.call(['python', 'scripts/answer_batch.py', '--merge', *(parts or [MERGED]), '--merge-out', MERGED])
save()
for s in SHARDS:
    print(f'\n===== Phần {s or "toàn bộ"}')
    print('\n'.join(open(log_of(s), encoding='utf-8').read().splitlines()[-6:]))
!ls -la {OUT}/outputs